# Week 1 Lab: Setup and the Course in Miniature
## Managing Semi-structured and Unstructured Data (ISYS1079 / ISYS3476)

## Problem setting

Structured data is organised into table fields that you can query with SQL. **In reviews, web pages and posts, much of the information is expressed in free text.** This course teaches two ways to work with that text:

> **Find** the text you need (search), then **make sense of it** (analyse with an LLM).

Today has two jobs: (1) set up your tools and claim your course LLM key, and (2) run the whole course *in miniature* so you can see where it's going: the first half (Assignment 1) builds the **find** side, the second half (Assignment 2) builds the **analyse** side. There is no graded coding this week.

## Setup (same on Mac and Windows)

0. **Unzip the course folder first.** Never work inside the zip.
1. Install **Python 3.10+** and **VS Code**. In VS Code, install the **Python** and **Jupyter** extensions.
2. Open VS Code, then choose **File -> Open Folder** and select the extracted practical folder. Open this notebook from the left file list.
3. Click **Select Kernel** at the top right and choose Python 3.10+. If the selected Python can run the notebook, continue to the first **install cell**. If VS Code says that `ipykernel` is required, click **Create a Python Environment**. Do not choose **Quick Create** if it shows Python 3.9. Choose **venv**, then choose a Python 3.10+ interpreter and select `requirements.txt` if prompted. If your newer Python is not listed, use **Enter Interpreter Path...**. Then select the new `.venv` kernel.

The normal setup does not require a terminal. `%pip` installs into the selected notebook kernel, which avoids the most common package and kernel mismatch.

| You see | Fix |
|---|---|
| Double-clicking the notebook opens something else | Open VS Code first, then File -> Open Folder |
| Popup: *Do you trust the authors?* | Click *Yes* |
| Popup: *ipykernel is required* | Click **Create a Python Environment** -> **venv** -> Python 3.10+ -> `requirements.txt`; do not use Quick Create if it shows Python 3.9 |
| `ModuleNotFoundError` after installing | Re-run the first install cell in this kernel |
| `externally-managed-environment` | Do not install into the system Python. Use **Create a Python Environment** -> **venv** as described above |
| Kernel list is empty | Install the Python and Jupyter extensions, then restart VS Code |
| Files or data not found | Extract the zip and reopen the extracted practical folder |


In [ ]:
# Run me first: install this semester's libraries into the selected Python 3.10+ kernel.
%pip install -q numpy nltk beautifulsoup4 whoosh-reloaded matplotlib openai ipykernel
import nltk
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)
print("Install finished. Now run the self-check cell below.")


In [ ]:
# Self-check: confirms which Python this notebook uses and that every library imports.
import sys
print("This notebook is using:", sys.executable)
ok = True
for lib in ["numpy", "nltk", "bs4", "whoosh", "matplotlib", "openai"]:
    try:
        m = __import__(lib)
        print(f"  {lib:16} OK ({getattr(m, '__version__', 'installed')})")
    except Exception as e:
        ok = False
        print(f"  {lib:16} MISSING -> re-run the install cell above in this kernel")
print("\nAll good, continue to Checkpoint 1." if ok else "\nFix the MISSING lines before continuing (see the table above).")

## Checkpoint 1: Vectors and cosine similarity

Search compares text as **vectors** and ranks by **cosine similarity** (from your linear algebra: about 1 = same direction, about 0 = unrelated). The vectors below are made-up numbers so you can see the score; Checkpoint 4 shows where real ones come from (the LLM turns text into them).

In [ ]:
import numpy as np
print("NumPy version:", np.__version__)

doc   = np.array([0.1, 0.5, 0.8, 0.2])
query = np.array([0.2, 0.4, 0.7, 0.3])

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Doc-query similarity: {cosine(doc, query):.3f}")

## Checkpoint 2: From text to tokens

Before text can become numbers it is split into **tokens** (roughly words), using NLTK. This is the first step of the first-half pipeline (Weeks 2-3).

In [ ]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize

print(word_tokenize("Search engines find relevant documents; LLMs help analyse them."))

## Checkpoint 3: Your course LLM access

**What this is:** this course gives you LLM access through a Canvas page called **LLM API Access** (under *Pages*). You claim a **personal API key** there once; it comes with a **Base URL** and a list of **models**, and you reuse it all semester (Weeks 7-10 and Assignment 2).

**Claim your key (do this now):**
1. Open the **LLM API Access** page on Canvas (https://rmit.instructure.com/courses/157398/pages/llm-api-access) and follow **Claim your API key**.
2. Enter the **course password** (shown on that Canvas page) to unlock the claim page, then your **student ID** (`e`/`s`, must be whitelisted) and the password *again* -> **Claim my API key**.
3. Copy your **Base URL**, **API Key**, and model names, and paste the first two below.

**Student ID not yet whitelisted?** Use the course fallback ID `e000000` for today's API checkpoint, and give your real student ID to your tutor so it can be added. The fallback key is shared and is only for Week 1 setup; switch to your own key once your ID is whitelisted.

**Chat models:** `gpt-5-nano`, `Ministral-3B`, `Phi-4-mini-instruct`. **Embedding model:** `text-embedding-3-small`. If a model returns a 404/unavailable error, switch to another (key & Base URL stay the same). Your current **budget** and remaining balance are shown on the claim page; keep `max_tokens` modest (output is capped at 512), but not tiny: `gpt-5-nano` uses a few tokens internally, so values like 5 can return an empty answer; 20+ is safe for a one-word label. **Never share or commit your key.**

In [ ]:
from openai import OpenAI

# TODO: paste these from the Canvas "LLM API Access" claim page (do NOT share your key)
API_BASE_URL = "TODO_PASTE_API_BASE_URL_HERE"
API_KEY      = "TODO_PASTE_API_KEY_HERE"
CHAT_MODEL   = "gpt-5-nano"
EMBED_MODEL  = "text-embedding-3-small"

client = OpenAI(api_key=API_KEY, base_url=API_BASE_URL)
print("Client configured. Chat model:", CHAT_MODEL)


## Checkpoint 4: First calls to the LLM

Two things the course API does for you:
- **Chat**: you send a question, it replies in text. This is the *analyse* half of the course.
- **Embedding**: you send text, it returns a *vector* (a list of numbers) that captures the meaning; similar meaning gives similar numbers. **This is where the made-up numbers in Checkpoint 1 come from, for real.**

Watch `response.usage`: every call costs tokens, and tokens cost budget.

In [ ]:
try:
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is information retrieval? Answer in 2 sentences."},
        ],
        max_tokens=120,
    )
    print(resp.choices[0].message.content.strip())
    print("\ntotal tokens:", resp.usage.total_tokens)
except Exception as e:
    print("Chat call FAILED:", e)
    print("Re-check API_BASE_URL and API_KEY from the LLM API Access page, the model name, and your internet.")


In [ ]:
emb = client.embeddings.create(model=EMBED_MODEL, input="information retrieval with language models")
vec = np.array(emb.data[0].embedding)
print("Embedding dimension:", vec.shape[0])
print("First 5 values:", [round(x, 4) for x in vec[:5]])
# This is the same kind of vector Checkpoint 1 compared with cosine similarity.

## Checkpoint 5: The course in miniature

Now put it together, for real:
1. **turn text into numbers**: embed the query and each review;
2. **find** the review most similar to the query: rank by cosine similarity, high to low (this is exactly what *search* does);
3. **make sense of it**: send the top review to the LLM for a label.

This `embed -> rank -> judge` is the whole course in one flow, and the shape of Assignment 2. **Provided:** the reviews and query below. **Try it yourself:** change the `query`, or add your own review to `docs`, and re-run.

In [ ]:
docs = [
    "The coffee was great but I waited 40 minutes for a flat white.",
    "Fast, friendly service and my order came out in two minutes.",
    "The library study rooms are quiet and easy to book online.",
    "Terrible queue, the staff ignored me and the latte was cold.",
]
query = "slow service and a long wait"

def embed_many(texts):
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in r.data])

doc_vecs = embed_many(docs)
q_vec    = embed_many([query])[0]

# score each review against the query, reusing cosine() from Checkpoint 1
sims   = [cosine(dv, q_vec) for dv in doc_vecs]
ranked = sorted(zip(sims, docs), reverse=True)   # highest similarity first

print("Query:", query, "\n")
for rank, (score, text) in enumerate(ranked, 1):
    print(f"{rank}. ({score:.3f})  {text}")

top_doc = ranked[0][1]

In [ ]:
# Analyse the top-ranked review with the LLM (the 'make sense of it' step)
resp = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "Label the cafe review as positive, neutral, or negative. Answer with one word only."},
        {"role": "user", "content": top_doc},
    ],
    max_tokens=20,  # gpt-5-nano spends a few tokens internally before the visible answer; values like 5 can return an empty string
)
print("Top review:", top_doc)
print("LLM label :", resp.choices[0].message.content.strip())

## Q&A and discussion

You just ran the course in miniature: **text -> vectors -> rank by similarity (find) -> ask an LLM to make sense of the top result (analyse).**

You do not need answers today; these are the questions the rest of the course works on. Discuss with the room, and bring questions to the teaching team:
1. Which review did the query match, and what made it win the ranking, even without sharing the exact words?
2. The LLM returned a one-word label. How would you check it is **correct**, not just well-formatted?
3. What would break if you ran this over **10,000** reviews instead of four: cost, speed, errors, aggregating the labels?
4. What text do **you** deal with day to day that has no schema (no neat table)?

**If your key or environment is not working yet, raise it now**: you need it before Week 7.

## Before next week
Week 2 is **data acquisition & preprocessing** (cleaning, tokenisation, building a reusable pipeline). Come with your environment working and your key claimed.